In [ ]:
#Tsfnet lstm eeg

In [1]:
#Tsfnet cross session
#version4
#preprocess corrected with session_id
#subject-level val + lr=0.0001 + weight_decay=0.1 + label_smoothing + two dense layers+ WEIGHT_DECAY_GRID = [0.01, 0.02, 0.05]
#Tsfnet model run h 11segments
# This is comprehensive TSFNet code, please set the paths basede on your system
import os
import gc
import math
# import numpy as np
# FIX (Problem 3, part 1): 'random' module was used below (random.shuffle) but never imported,
# which would raise NameError: name 'random' is not defined the moment the script runs.
import random  # NEW: was missing, required for random.shuffle below
import numpy as np  # NEW: moved down here only for readability, same import as before
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import classification_report, accuracy_score
# NEW (user request): confusion_matrix needed for the detailed error-pattern analysis, and
# accuracy_score is reused per-branch for the EEG-only / fNIRS-only ablation below.
from sklearn.metrics import confusion_matrix  # NEW
from sklearn.metrics import cohen_kappa_score  # NEW: adopted from reviewing a third-party implementation -- corrects raw accuracy for chance agreement, a standard BCI-literature metric
import warnings
import json  # NEW: for results.json export, adopted from reviewing a third-party implementation
warnings.filterwarnings('ignore')

# CONFIGURATION
PROCESSED_DATA_PATH = r"F:\preprocessing-history-thesis\preproc_P2_no_fnirs_filter_session_id in code at the same time"  # FIX: was E:\, change to match your actual drive

ALL_SUBJECTS = list(range(1, 27))

RANDOM_SEED = 42  # fixed seed for reproducible model init and window-level val split
random.seed(RANDOM_SEED)
torch.manual_seed(RANDOM_SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False
np.random.seed(RANDOM_SEED)

class MMapHybridDataset(Dataset): 

    def __init__(self, subs_list, dataset_name="Dataset",
                 session_filter=None,        # CROSS-SESSION: None=همه sessions، [1,2]=فقط session 1 و 2
                 min_windows_threshold=500):
        super().__init__()
        self.dataset_name = dataset_name
        self.session_filter = session_filter  # CROSS-SESSION: کدام session‌ها بارگذاری شوند
        self.valid_files = []
        self.total_windows = 0
        
        filter_str = f"sessions {session_filter}" if session_filter is not None else "all sessions"
        print(f"Scanning {dataset_name} - {len(subs_list)} subjects ({filter_str})...")
        
        if not os.path.exists(PROCESSED_DATA_PATH):
            raise FileNotFoundError(
                f"\n[PATH ERROR] PROCESSED_DATA_PATH does not exist:\n"
                f"  '{PROCESSED_DATA_PATH}'\n"
                f"  Check your drive letter (D: vs E:) and folder structure.\n"
                f"  Expected subject folders: VP001, VP002, ... inside that directory."
            )
        
        for sub_id in subs_list:
            sub_str = f"{sub_id:03d}"
            subject = f"VP{sub_str}"
            sub_folder = os.path.join(PROCESSED_DATA_PATH, subject)
            
            if not os.path.exists(sub_folder):
                continue
                
            eeg_file = next((f for f in [
                os.path.join(sub_folder, f"{subject}_eeg_windows.npy"), 
                os.path.join(sub_folder, f"{subject}_eeg.npy")
            ] if os.path.exists(f)), None)
            
            nirs_file = next((f for f in [
                os.path.join(sub_folder, f"{subject}_fnirs_windows.npy"), 
                os.path.join(sub_folder, f"{subject}_fnirs.npy")
            ] if os.path.exists(f)), None)
            
            labels_file = os.path.join(sub_folder, f"{subject}_labels.npy")
            
            if eeg_file and nirs_file and os.path.exists(labels_file):
                try:
                    lbls = np.load(labels_file).astype(np.int64)

                    session_mask = None
                    if session_filter is not None:
                        session_ids_file = os.path.join(sub_folder, f"{subject}_session_ids.npy")
                        if os.path.exists(session_ids_file):
                            sess_ids = np.load(session_ids_file)
                            session_mask = np.isin(sess_ids, session_filter)
                            lbls = lbls[session_mask]
                        else:
                            print(f"  WARNING: {subject}_session_ids.npy not found — using all windows")

                    n_windows = lbls.shape[0]
                    
                    if n_windows < min_windows_threshold:
                        print(f"{subject} Skipped - Insufficient windows ({n_windows} < {min_windows_threshold}). Check task markers!")
                        continue
                    
                    self.valid_files.append({
                        'subject': subject,
                        'eeg_path': eeg_file,
                        'nirs_path': nirs_file,
                        'labels': lbls,
                        'session_mask': session_mask,
                        'start_idx': self.total_windows,
                        'end_idx': self.total_windows + n_windows
                    })
                    self.total_windows += n_windows
                    print(f"{subject} verified: {n_windows} windows")
                    
                except Exception as e:
                    print(f"Error scanning {subject}: {str(e)}")
                    continue
                    
        if self.total_windows == 0:
            raise ValueError(f"No valid preprocessed data found for {dataset_name} after filtering!")

        os.makedirs(PROCESSED_DATA_PATH, exist_ok=True)

        import re
        safe_slug = re.sub(r'[^a-z0-9]', '_', dataset_name.lower())[:12]
        self.mmap_eeg_path  = os.path.join(PROCESSED_DATA_PATH, f"t_mmap_{safe_slug}_e.npy")
        self.mmap_nirs_path = os.path.join(PROCESSED_DATA_PATH, f"t_mmap_{safe_slug}_n.npy")
        
        for f_path in [self.mmap_eeg_path, self.mmap_nirs_path]:
            if os.path.exists(f_path):
                try: os.remove(f_path)
                except: pass
            with open(f_path, 'wb') as f:
                pass 
        
        self.X_eeg = np.memmap(self.mmap_eeg_path, dtype='float32', mode='w+', shape=(self.total_windows, 16, 16, 600))
        self.X_nirs = np.memmap(self.mmap_nirs_path, dtype='float32', mode='w+', shape=(self.total_windows, 11, 16, 16, 30, 2))
        self.y = np.zeros(self.total_windows, dtype='int64')
        
        print(f"Streaming clean data to disk memory-map ({dataset_name})...")
        for f_info in self.valid_files:
            s, e = f_info['start_idx'], f_info['end_idx']
            eeg_full  = np.load(f_info['eeg_path'])
            nirs_full = np.load(f_info['nirs_path'])
            if f_info['session_mask'] is not None:
                eeg_full  = eeg_full[f_info['session_mask']]
                nirs_full = nirs_full[f_info['session_mask']]
            self.X_eeg[s:e]  = eeg_full
            self.X_nirs[s:e] = nirs_full
            self.y[s:e]      = f_info['labels']
            gc.collect()
            
        self.X_eeg.flush()
        self.X_nirs.flush()
        print(f"--> {dataset_name} Ready on Disk (CLEAN)! Total Windows: {self.total_windows} | Labels: {np.bincount(self.y)}\n")

    def __len__(self):
        return self.total_windows

    def __getitem__(self, idx):
        eeg_tensor = torch.from_numpy(self.X_eeg[idx]).float()
        nirs_tensor = torch.from_numpy(self.X_nirs[idx]).float()
        label_tensor = torch.tensor(self.y[idx], dtype=torch.long)
        return eeg_tensor, nirs_tensor, label_tensor
        
    def cleanup(self):
        del self.X_eeg
        del self.X_nirs
        gc.collect()
        for f in [self.mmap_eeg_path, self.mmap_nirs_path]:
            if os.path.exists(f):
                try: os.remove(f)
                except: pass

def pearson_correlation_loss(a, b, eps=1e-8):
    a_flat = a.reshape(-1)
    b_flat = b.reshape(-1)
    a_centered = a_flat - a_flat.mean()
    b_centered = b_flat - b_flat.mean()
    numerator = (a_centered * b_centered).sum()
    denominator = torch.sqrt((a_centered ** 2).sum() * (b_centered ** 2).sum()) + eps
    pcc = numerator / denominator
    return 1.0 - torch.abs(pcc)


class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=100):
        super().__init__()
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        self.register_buffer('pe', pe.unsqueeze(0))
    def forward(self, x): 
        return x + self.pe[:, :x.size(1), :]

class CrossAttention(nn.Module):
    def __init__(self, d_model=256, nhead=8, dropout=0.5, dim_feedforward=512):
        super().__init__()
        self.attn     = nn.MultiheadAttention(d_model, nhead, dropout=dropout, batch_first=True)
        self.norm1    = nn.LayerNorm(d_model)
        self.norm2    = nn.LayerNorm(d_model)
        self.dropout1 = nn.Dropout(dropout)
        self.dropout2 = nn.Dropout(dropout)
        self.ffn = nn.Sequential(
            nn.Linear(d_model, dim_feedforward),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(dim_feedforward, d_model),
        )
    def forward(self, query, key, value):
        attn_out, _ = self.attn(query, key, value)
        x = self.norm1(query + self.dropout1(attn_out))
        x = self.norm2(x + self.dropout2(self.ffn(x)))
        return x

class TSFNet_With_fNIRS_2D(nn.Module):
    def __init__(self, eeg_ch=28, num_classes=3, dropout=0.5):
        super(TSFNet_With_fNIRS_2D, self).__init__()

        # EEG branch
        self.eeg_conv1 = nn.Conv3d(1, 16, kernel_size=(12, 4, 4), stride=(6, 2, 2), padding=(3, 1, 1))
        self.eeg_bn1 = nn.BatchNorm3d(16)
        self.eeg_drop1 = nn.Dropout(dropout)
        self.eeg_conv2 = nn.Conv3d(16, 32, kernel_size=(6, 2, 2), stride=(2, 2, 2), padding=(2, 0, 0))
        self.eeg_bn2 = nn.BatchNorm3d(32)
        self.eeg_drop2 = nn.Dropout(dropout)
        self.eeg_fc = nn.Linear(32 * 4 * 4, 64)

        # NEW (Mughal et al. 2022-inspired Time-Distributed CNN-LSTM): replaces the plain temporal
        # GAP used for the EEG classifier head only. eeg_feat_map (mean over time) is still computed
        # separately and unchanged for L_EFGF -- this LSTM path is isolated and does not affect it.
        # input_size = flattened spatial dims of ex2 per timestep (32 channels * 4 * 4 spatial).
        self.eeg_temporal_lstm = nn.LSTM(
            input_size=32 * 4 * 4,
            hidden_size=64,
            num_layers=1,
            batch_first=True,
            bidirectional=False
        )  # NEW
        self.eeg_fc_from_lstm = nn.Linear(64, 64)  # NEW

        # fNIRS branch
        self.nirs_conv1 = nn.Conv3d(2, 16, kernel_size=(6, 4, 4), stride=(2, 2, 2), padding=(2, 1, 1))
        self.nirs_bn1 = nn.BatchNorm3d(16)
        self.nirs_drop1 = nn.Dropout(dropout)
        self.nirs_conv2 = nn.Conv3d(16, 32, kernel_size=(3, 2, 2), stride=(2, 2, 2), padding=(1, 0, 0))
        self.nirs_bn2 = nn.BatchNorm3d(32)
        self.nirs_drop2 = nn.Dropout(dropout)
        self.nirs_fc = nn.Linear(32 * 4 * 4, 64)

        # Fusion branch
        self.fus_conv1 = nn.Conv3d(1, 16, kernel_size=(12, 4, 4), stride=(6, 2, 2), padding=(3, 1, 1))
        self.fus_bn1   = nn.BatchNorm3d(16)
        self.fus_drop1 = nn.Dropout(dropout)
        self.fus_conv2 = nn.Conv3d(16, 32, kernel_size=(6, 2, 2), stride=(2, 2, 2), padding=(2, 0, 0))
        self.fus_bn2   = nn.BatchNorm3d(32)
        self.fus_drop2 = nn.Dropout(dropout)

        # EFGF two-stage
        self.efgf1_eeg_conv3d  = nn.Conv3d(16, 1, kernel_size=(6, 2, 2), stride=(1, 1, 1), padding=(0, 0, 0))
        self.efgf1_nirs_conv3d = nn.Conv3d(16, 1, kernel_size=(6, 2, 2), stride=(1, 1, 1), padding=(0, 0, 0))
        self.alpha1 = nn.Parameter(torch.tensor(0.0))
        self.gamma1 = nn.Parameter(torch.tensor(0.0))

        self.efgf2_eeg_conv3d  = nn.Conv3d(32, 1, kernel_size=(3, 2, 2), stride=(1, 1, 1), padding=(0, 0, 0))
        self.efgf2_nirs_conv3d = nn.Conv3d(32, 1, kernel_size=(3, 2, 2), stride=(1, 1, 1), padding=(0, 0, 0))
        self.alpha2 = nn.Parameter(torch.tensor(0.0))
        self.gamma2 = nn.Parameter(torch.tensor(0.0))

        self.fusion_proj = nn.Linear(32 * 4 * 4, 64)

        # CAFE
        CAFE_DIM = 128
        self.pos_encoder      = PositionalEncoding(CAFE_DIM, max_len=11)
        self.nirs_seq_proj    = nn.Linear(32, CAFE_DIM)
        self.fusion_cafe_proj = nn.Linear(64, CAFE_DIM)
        self.nirs_cafe_proj   = nn.Linear(64, CAFE_DIM)
        self.cafe_attn  = CrossAttention(CAFE_DIM, nhead=8, dropout=dropout, dim_feedforward=256)
        self.cafe_gate  = nn.Linear(CAFE_DIM * 2, CAFE_DIM)
        self.cafe_attn2 = CrossAttention(CAFE_DIM, nhead=8, dropout=dropout, dim_feedforward=256)
        self.cafe_gate2 = nn.Linear(CAFE_DIM * 2, CAFE_DIM)
        self.cafe_out_proj  = nn.Linear(CAFE_DIM, 64)
        self.cafe_out_proj2 = nn.Linear(CAFE_DIM, 64)

        # classifiers
        self.fc_eeg = nn.Sequential(
            nn.Linear(64, 32), nn.ELU(), nn.Dropout(dropout),
            nn.Linear(32, num_classes)
        )
        self.fc_fnirs = nn.Sequential(
            nn.Linear(64, 32), nn.ELU(), nn.Dropout(dropout),
            nn.Linear(32, num_classes)
        )
        self.fc_fusion = nn.Sequential(
            nn.Linear(64, 32), nn.ELU(), nn.Dropout(dropout),
            nn.Linear(32, num_classes)
        )

        self.w1 = nn.Parameter(torch.tensor(1.0))
        self.w2 = nn.Parameter(torch.tensor(1.0))
        self.w3 = nn.Parameter(torch.tensor(1.0))
        self.dropout = nn.Dropout(dropout)
        
    def forward(self, eeg, nirs):
        batch_size = eeg.size(0)

        # EEG branch
        eeg_in = eeg.permute(0, 3, 1, 2).unsqueeze(1)       # (batch,1,600,16,16)
        ex1 = F.elu(self.eeg_bn1(self.eeg_conv1(eeg_in)))   # (batch,16,100,8,8)
        ex1 = self.eeg_drop1(ex1)
        ex2 = F.elu(self.eeg_bn2(self.eeg_conv2(ex1)))      # (batch,32,50,4,4)
        ex2 = self.eeg_drop2(ex2)

        # GAP path (unchanged, isolated) -- used ONLY for L_EFGF, not for the EEG classifier anymore
        eeg_feat_map = ex2.mean(dim=2)                       # (batch,32,4,4) — برای L_EFGF فقط

        # NEW: LSTM path for EEG classifier head — replaces GAP-based eeg_feat.
        # .detach() on ex2 ensures eeg_conv1/eeg_conv2 receive NO gradient from this LSTM path;
        # they are only trained through the fusion branch / EFGF path, exactly like eeg_feat_map
        # above and consistent with the prior SpatialTransformer isolation lesson.
        batch_size_eeg = ex2.size(0)
        T_dim = ex2.size(2)
        eeg_seq = ex2.detach().permute(0, 2, 1, 3, 4).reshape(batch_size_eeg, T_dim, -1)  # NEW: (batch,T,512)
        lstm_out, (h_n, c_n) = self.eeg_temporal_lstm(eeg_seq)  # NEW
        eeg_feat_from_lstm = h_n.squeeze(0)                      # NEW: (batch,64)
        eeg_feat = self.eeg_fc_from_lstm(eeg_feat_from_lstm)     # NEW: (batch,64)

        # Fusion branch — Conv1 (قبل از EFGF1)
        fx1 = F.elu(self.fus_bn1(self.fus_conv1(eeg_in)))   # (batch,16,100,8,8)
        fx1 = self.fus_drop1(fx1)

        # fNIRS branch
        n_seg  = nirs.size(1)
        nirs_r = nirs.permute(0, 1, 5, 4, 2, 3)
        nirs_r = nirs_r.reshape(batch_size * n_seg, 2, 30, 16, 16)
        ny1 = F.elu(self.nirs_bn1(self.nirs_conv1(nirs_r)))
        ny1 = self.nirs_drop1(ny1)
        ny2 = F.elu(self.nirs_bn2(self.nirs_conv2(ny1)))
        ny2 = self.nirs_drop2(ny2)
        ny1 = ny1.reshape(batch_size, n_seg, 16, 15, 8, 8)
        ny2 = ny2.reshape(batch_size, n_seg, 32,  8, 4, 4)
        nirs_feat_map = ny2.mean(dim=(1, 3))
        nirs_feat = self.nirs_fc(nirs_feat_map.flatten(1))
        y_spatial = ny2.mean(dim=(3, 4, 5))

        # EFGF1
        ny1_vol = ny1.mean(dim=1)
        eeg_a1_vol  = self.efgf1_eeg_conv3d(ex1)
        nirs_a1_vol = self.efgf1_nirs_conv3d(ny1_vol)
        eeg_a1  = torch.sigmoid(eeg_a1_vol.mean(dim=2))
        nirs_a1 = torch.sigmoid(nirs_a1_vol.mean(dim=2))
        a1   = torch.clamp(self.alpha1, 0, 1)
        Phi1 = a1 * nirs_a1 + (1 - a1) * eeg_a1
        g1 = torch.clamp(self.gamma1, 0, 1)
        Phi1_bc = Phi1.unsqueeze(2)
        fx1_updated = g1 * ex1[:, :, :, :Phi1.shape[-2], :Phi1.shape[-1]] \
                    + (1 - g1) * fx1[:, :, :, :Phi1.shape[-2], :Phi1.shape[-1]] \
                    + Phi1_bc * fx1[:, :, :, :Phi1.shape[-2], :Phi1.shape[-1]]

        if fx1_updated.shape != fx1.shape:
            fx1_updated_aligned = F.adaptive_avg_pool3d(
                fx1_updated, (fx1.shape[2], fx1.shape[3], fx1.shape[4])
            )
        else:
            fx1_updated_aligned = fx1_updated
        fx2 = F.elu(self.fus_bn2(self.fus_conv2(fx1_updated_aligned)))
        fx2 = self.fus_drop2(fx2)

        # EFGF2
        ny2_vol = ny2.mean(dim=1)
        eeg_a2_vol  = self.efgf2_eeg_conv3d(ex2)
        nirs_a2_vol = self.efgf2_nirs_conv3d(ny2_vol)
        eeg_a2  = torch.sigmoid(eeg_a2_vol.mean(dim=2))
        nirs_a2 = torch.sigmoid(nirs_a2_vol.mean(dim=2))
        a2   = torch.clamp(self.alpha2, 0, 1)
        Phi2 = a2 * nirs_a2 + (1 - a2) * eeg_a2
        g2   = torch.clamp(self.gamma2, 0, 1)
        Phi2_bc = Phi2.unsqueeze(2)
        sh, sw = Phi2.shape[-2], Phi2.shape[-1]
        fx2_updated = g2 * ex2[:, :, :, :sh, :sw] \
                    + (1 - g2) * fx2[:, :, :, :sh, :sw] \
                    + Phi2_bc * fx2[:, :, :, :sh, :sw]
        fus_feat_map = fx2_updated.mean(dim=2)
        fus_feat_map = F.adaptive_avg_pool2d(fus_feat_map, (4, 4))
        fused_feat   = self.fusion_proj(fus_feat_map.flatten(1))

        Phi2_bc_loss = Phi2.expand(-1, 32, -1, -1)
        if Phi2_bc_loss.shape != eeg_feat_map.shape:
            Phi2_bc_loss = F.adaptive_avg_pool2d(Phi2_bc_loss, eeg_feat_map.shape[-2:])

        # CAFE
        nirs_seq_256  = self.nirs_seq_proj(y_spatial)
        nirs_proj_256 = self.pos_encoder(nirs_seq_256)
        fused_256     = self.fusion_cafe_proj(fused_feat)
        nirs_feat_256 = self.nirs_cafe_proj(nirs_feat)
        fused_seq_256  = fused_256.unsqueeze(1)
        enhanced_256   = self.cafe_attn(fused_seq_256, nirs_proj_256, nirs_proj_256).squeeze(1)
        gate2_256      = torch.sigmoid(self.cafe_gate(torch.cat([fused_256, enhanced_256], dim=-1)))
        final_feat_256 = gate2_256 * enhanced_256 + (1 - gate2_256) * fused_256
        nirs_q_256     = nirs_feat_256.unsqueeze(1)
        fused_kv_256   = fused_256.unsqueeze(1)
        enh_nirs_256   = self.cafe_attn2(nirs_q_256, fused_kv_256, fused_kv_256).squeeze(1)
        gate3_256      = torch.sigmoid(self.cafe_gate2(torch.cat([nirs_feat_256, enh_nirs_256], dim=-1)))
        final_nirs_256 = gate3_256 * enh_nirs_256 + (1 - gate3_256) * nirs_feat_256
        final_feat = self.cafe_out_proj(final_feat_256)
        final_nirs = self.cafe_out_proj2(final_nirs_256)

        # Classifier
        eeg_out    = self.fc_eeg(self.dropout(eeg_feat))
        fnirs_out  = self.fc_fnirs(self.dropout(final_nirs))
        fusion_out = self.fc_fusion(self.dropout(final_feat))

        w1 = torch.sigmoid(self.w1)
        w2 = torch.sigmoid(self.w2)
        w3 = torch.sigmoid(self.w3)
        final_out = w1 * eeg_out + w2 * fusion_out + w3 * fnirs_out

        return (final_out, eeg_out, fnirs_out, fusion_out,
                eeg_feat_map, Phi2_bc_loss,
                final_nirs,
                final_feat)


LAMBDA_EEG = 0.2
LAMBDA_FNIRS = 0.2
LAMBDA_EFGF_REG = 1.0
LAMBDA_CAFE_REG = 1.0

STAGE1_MAX_EPOCHS = 300
STAGE1_PATIENCE   = 50
STAGE2_MAX_EPOCHS = 200


def run_training(weight_decay, device, test_session=3):
    train_sessions = [s for s in [1, 2, 3] if s != test_session]
    print("\n" + "#" * 70)
    print(f"# CROSS-SESSION: train on sessions {train_sessions}  |  test on session {test_session}")
    print("#" * 70)

    from torch.utils.data import SubsetRandomSampler

    train_dataset = MMapHybridDataset(ALL_SUBJECTS, "Train_Set",
                                       session_filter=train_sessions,
                                       min_windows_threshold=200)
    test_dataset  = MMapHybridDataset(ALL_SUBJECTS, "Test_Set",
                                       session_filter=[test_session],
                                       min_windows_threshold=100)

    all_trial_ids_global = []
    window_offset = 0
    for f_info in train_dataset.valid_files:
        subject = f_info['subject']
        sub_folder = os.path.join(PROCESSED_DATA_PATH, subject)
        trial_ids_path = os.path.join(sub_folder, f"{subject}_trial_ids.npy")
        
        trial_ids_full = np.load(trial_ids_path)
        
        if f_info['session_mask'] is not None:
            trial_ids_filtered = trial_ids_full[f_info['session_mask']]
        else:
            trial_ids_filtered = trial_ids_full
        
        n_windows_this_subject = f_info['end_idx'] - f_info['start_idx']
        assert len(trial_ids_filtered) == n_windows_this_subject, \
            f"{subject}: trial_ids mismatch ({len(trial_ids_filtered)} vs {n_windows_this_subject})"
        
        sub_idx = ALL_SUBJECTS.index(int(subject[2:]))
        global_trial_ids = sub_idx * 100 + trial_ids_filtered
        all_trial_ids_global.extend(global_trial_ids.tolist())
        window_offset += n_windows_this_subject
    
    all_trial_ids_global = np.array(all_trial_ids_global)
    assert len(all_trial_ids_global) == len(train_dataset), \
        f"trial_ids length mismatch: {len(all_trial_ids_global)} vs {len(train_dataset)}"
    
    unique_trials = np.unique(all_trial_ids_global)
    rng = np.random.default_rng(RANDOM_SEED)
    rng.shuffle(unique_trials)
    
    split_trial = int(len(unique_trials) * 0.8)
    train_trials = set(unique_trials[:split_trial])
    val_trials   = set(unique_trials[split_trial:])
    
    s1_train_indices = [i for i, t in enumerate(all_trial_ids_global) if t in train_trials]
    s1_val_indices   = [i for i, t in enumerate(all_trial_ids_global) if t in val_trials]
    
    print(f"[Trial-level split] {len(unique_trials)} trials total | "
          f"{len(train_trials)} train trials ({len(s1_train_indices)} windows) | "
          f"{len(val_trials)} val trials ({len(s1_val_indices)} windows) | "
          f"ratio={len(s1_train_indices)/max(len(s1_val_indices),1):.1f}:1")
    print(f"[Trial-level split] Zero overlap guaranteed: "
          f"no window from a val trial appears in train")

    s1_train_loader = DataLoader(
        train_dataset, batch_size=64,
        sampler=SubsetRandomSampler(s1_train_indices),
        num_workers=0
    )
    s1_val_loader = DataLoader(
        train_dataset, batch_size=64,
        sampler=SubsetRandomSampler(s1_val_indices),
        num_workers=0
    )
    train_loader  = DataLoader(train_dataset, batch_size=64, shuffle=True,  num_workers=0)
    test_loader   = DataLoader(test_dataset,  batch_size=64, shuffle=False, num_workers=0)

    model = TSFNet_With_fNIRS_2D(eeg_ch=28, num_classes=3, dropout=0.5).to(device)

    print(f"Model Initialized. Total Parameters: {sum(p.numel() for p in model.parameters()):,}")
    print(f"Running on Device: {device}\n")

    optimizer = optim.Adam(model.parameters(), lr=0.001)

    _train_labels   = train_dataset.y
    _class_counts   = np.bincount(_train_labels, minlength=3).astype(float)
    _class_weights  = 1.0 / (_class_counts + 1e-6)
    _class_weights  = _class_weights / _class_weights.sum() * 3
    _weight_tensor  = torch.tensor(_class_weights, dtype=torch.float32).to(device)
    print(f"[3-class] Class counts (train): 0-back={int(_class_counts[0])}, "
          f"2-back={int(_class_counts[1])}, 3-back={int(_class_counts[2])}")
    print(f"[3-class] Class weights:        0-back={_class_weights[0]:.3f}, "
          f"2-back={_class_weights[1]:.3f}, 3-back={_class_weights[2]:.3f}")
    criterion = nn.CrossEntropyLoss(weight=_weight_tensor)

    ckpt_path = os.path.join(PROCESSED_DATA_PATH, f'best_tsfnet_session{test_session}.pth')

    print(f"STAGE 1 — training on {len(s1_train_indices)} windows (80% of train sessions {train_sessions}), "
          f"val on {len(s1_val_indices)} windows (20%) | "
          f"max {STAGE1_MAX_EPOCHS} epochs | patience {STAGE1_PATIENCE} | trial-level 4:1 split (paper)")

    best_val_acc     = 0.0
    epochs_no_improve = 0
    stage1_best_val_train_loss = float('inf')
    stage1_final_train_loss    = float('inf')
    train_acc = 0.0

    for epoch in range(1, STAGE1_MAX_EPOCHS + 1):
        model.train()
        train_loss, train_class_loss, train_efgf_loss, train_cafe_loss, train_correct = 0.0, 0.0, 0.0, 0.0, 0

        for eeg_b, nirs_b, lbl_b in s1_train_loader:
            eeg_b, nirs_b, lbl_b = eeg_b.to(device), nirs_b.to(device), lbl_b.to(device)
            optimizer.zero_grad()
            pred, eeg_out, fnirs_out, fusion_out, eeg_feat_map, efgf_attn_map, cafe_fnirs_feat, cafe_fusion_feat = model(eeg_b, nirs_b)

            L_pred   = criterion(pred,      lbl_b)
            L_eeg    = criterion(eeg_out,   lbl_b)
            L_fnirs  = criterion(fnirs_out, lbl_b)
            L_fusion = criterion(fusion_out,lbl_b)
            L_class  = L_pred + LAMBDA_EEG * L_eeg + LAMBDA_FNIRS * L_fnirs + L_fusion
            L_efgf   = pearson_correlation_loss(eeg_feat_map, efgf_attn_map)
            L_cafe   = pearson_correlation_loss(cafe_fnirs_feat, cafe_fusion_feat)
            loss     = L_class + LAMBDA_EFGF_REG * L_efgf + LAMBDA_CAFE_REG * L_cafe

            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()

            train_loss       += loss.item()
            train_class_loss += L_class.item()
            train_efgf_loss  += L_efgf.item()
            train_cafe_loss  += L_cafe.item()
            train_correct    += (pred.argmax(1) == lbl_b).sum().item()

        train_acc      = train_correct / len(s1_train_indices)
        avg_train_loss = train_loss / len(s1_train_loader)
        stage1_final_train_loss = avg_train_loss

        model.eval()
        val_correct = 0
        with torch.no_grad():
            for eeg_b, nirs_b, lbl_b in s1_val_loader:
                eeg_b, nirs_b, lbl_b = eeg_b.to(device), nirs_b.to(device), lbl_b.to(device)
                pred, _, _, _, _, _, _, _ = model(eeg_b, nirs_b)
                val_correct += (pred.argmax(1) == lbl_b).sum().item()
        val_acc = val_correct / len(s1_val_indices)

        if val_acc > best_val_acc:
            best_val_acc = val_acc
            stage1_best_val_train_loss = avg_train_loss
            torch.save(model.state_dict(), ckpt_path)
            epochs_no_improve = 0
        else:
            epochs_no_improve += 1

        if epoch % 5 == 0 or epoch == 1:
            n_batches = len(s1_train_loader)
            print(f"[S1] Epoch {epoch:3d}/{STAGE1_MAX_EPOCHS} | Loss: {avg_train_loss:.4f} | "
                  f"Class: {train_class_loss/n_batches:.4f} | "
                  f"EFGF: {train_efgf_loss/n_batches:.4f} | "
                  f"CAFE: {train_cafe_loss/n_batches:.4f} | "
                  f"Train Acc: {train_acc*100:.2f}% | Val Acc: {val_acc*100:.2f}% | Best Val: {best_val_acc*100:.2f}%")

        if epochs_no_improve >= STAGE1_PATIENCE:
            print(f"\n[S1] Early stopping at epoch {epoch} "
                  f"(no val improvement for {STAGE1_PATIENCE} epochs). "
                  f"Best Val: {best_val_acc*100:.2f}%")
            break

    model.load_state_dict(torch.load(ckpt_path))
    stage2_target = min(stage1_best_val_train_loss, stage1_final_train_loss)
    print(f"\n[S1] Best checkpoint restored.")
    print(f"[S1] best-val-epoch loss: {stage1_best_val_train_loss:.4f} | final-epoch loss: {stage1_final_train_loss:.4f}")
    print(f"[S1] Stage-2 target loss: {stage2_target:.4f} (min of the two — ensures stage 2 actually trains)")

    print(f"\nSTAGE 2 — retraining on full {len(train_dataset)} samples ({len(ALL_SUBJECTS)} subjects, sessions {train_sessions}) "
          f"| max {STAGE2_MAX_EPOCHS} epochs | target loss < {stage2_target:.4f}")

    optimizer2 = optim.Adam(model.parameters(), lr=0.001)

    for epoch in range(1, STAGE2_MAX_EPOCHS + 1):
        model.train()
        s2_loss, s2_correct = 0.0, 0
        s2_class_loss = s2_efgf_loss = s2_cafe_loss = 0.0

        for eeg_b, nirs_b, lbl_b in train_loader:
            eeg_b, nirs_b, lbl_b = eeg_b.to(device), nirs_b.to(device), lbl_b.to(device)
            optimizer2.zero_grad()
            pred, eeg_out, fnirs_out, fusion_out, eeg_feat_map, efgf_attn_map, cafe_fnirs_feat, cafe_fusion_feat = model(eeg_b, nirs_b)

            L_pred   = criterion(pred,      lbl_b)
            L_eeg    = criterion(eeg_out,   lbl_b)
            L_fnirs  = criterion(fnirs_out, lbl_b)
            L_fusion = criterion(fusion_out,lbl_b)
            L_class  = L_pred + LAMBDA_EEG * L_eeg + LAMBDA_FNIRS * L_fnirs + L_fusion
            L_efgf   = pearson_correlation_loss(eeg_feat_map, efgf_attn_map)
            L_cafe   = pearson_correlation_loss(cafe_fnirs_feat, cafe_fusion_feat)
            loss     = L_class + LAMBDA_EFGF_REG * L_efgf + LAMBDA_CAFE_REG * L_cafe

            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer2.step()

            s2_loss       += loss.item()
            s2_class_loss += L_class.item()
            s2_efgf_loss  += L_efgf.item()
            s2_cafe_loss  += L_cafe.item()
            s2_correct    += (pred.argmax(1) == lbl_b).sum().item()

        avg_s2_loss = s2_loss / len(train_loader)
        train_acc   = s2_correct / len(train_loader.dataset)

        if epoch % 5 == 0 or epoch == 1:
            n_b = len(train_loader)
            print(f"[S2] Epoch {epoch:3d}/{STAGE2_MAX_EPOCHS} | Loss: {avg_s2_loss:.4f} | "
                  f"Class: {s2_class_loss/n_b:.4f} | "
                  f"EFGF: {s2_efgf_loss/n_b:.4f} | "
                  f"CAFE: {s2_cafe_loss/n_b:.4f} | "
                  f"Train Acc: {train_acc*100:.2f}% | Target loss: {stage2_target:.4f}")

        if avg_s2_loss < stage2_target:
            print(f"\n[S2] Stopping at epoch {epoch}: train loss {avg_s2_loss:.4f} "
                  f"< target {stage2_target:.4f}")
            torch.save(model.state_dict(), ckpt_path)
            break
    else:
        torch.save(model.state_dict(), ckpt_path)
        print(f"[S2] Reached max epochs ({STAGE2_MAX_EPOCHS}) without meeting loss target. Saving final weights.")

    print(f"\n RUNNING FINAL TEST EVALUATION (test_session={test_session})")
    model.load_state_dict(torch.load(ckpt_path))
    model.eval()

    all_preds_final, all_preds_eeg, all_preds_fnirs, all_preds_fusion, all_true = [], [], [], [], []
    with torch.no_grad():
        for eeg_b, nirs_b, lbl_b in test_loader:
            eeg_b, nirs_b, lbl_b = eeg_b.to(device), nirs_b.to(device), lbl_b.to(device)
            pred, eeg_out, fnirs_out, fusion_out, _, _, _, _ = model(eeg_b, nirs_b)
            all_preds_final.extend(pred.argmax(1).cpu().numpy())
            all_preds_eeg.extend(eeg_out.argmax(1).cpu().numpy())
            all_preds_fnirs.extend(fnirs_out.argmax(1).cpu().numpy())
            all_preds_fusion.extend(fusion_out.argmax(1).cpu().numpy())
            all_true.extend(lbl_b.cpu().numpy())

    class_names = ['0-back', '2-back', '3-back']

    test_acc = accuracy_score(all_true, all_preds_final) * 100
    kappa_final = cohen_kappa_score(all_true, all_preds_final)
    print(f"\nOverall Model Test Accuracy (final fused prediction): {test_acc:.2f}%")
    print(f"Kappa Score (final fused prediction): {kappa_final:.4f}")
    print("\nClassification Report (final fused prediction):")
    print(classification_report(all_true, all_preds_final, target_names=class_names))

    cm = confusion_matrix(all_true, all_preds_final)
    print("\nConfusion Matrix (rows=true, cols=predicted):")
    header = "        " + "".join([f"{c:>10}" for c in class_names])
    print(header)
    for i, row in enumerate(cm):
        row_str = "".join([f"{v:>10}" for v in row])
        print(f"{class_names[i]:>8}{row_str}")

    eeg_acc = accuracy_score(all_true, all_preds_eeg) * 100
    fnirs_acc = accuracy_score(all_true, all_preds_fnirs) * 100
    fusion_acc = accuracy_score(all_true, all_preds_fusion) * 100
    print("\nZero-Retraining Ablation (individual classifier heads of the SAME trained model):")
    print(f"  EEG-branch-only accuracy:          {eeg_acc:.2f}%")
    print(f"  fNIRS-branch-only accuracy:         {fnirs_acc:.2f}%")
    print(f"  EFGF+CAFE fused-feature accuracy:   {fusion_acc:.2f}%")
    print(f"  Final weighted-combination accuracy: {test_acc:.2f}%  (Mean(w1*eeg, w2*fusion, w3*fnirs))")

    with torch.no_grad():
        learned_w1 = torch.sigmoid(model.w1).item()
        learned_w2 = torch.sigmoid(model.w2).item()
        learned_w3 = torch.sigmoid(model.w3).item()
    print(f"\nLearned final-combination weights (after sigmoid, paper Eq.4): "
          f"w_eeg={learned_w1:.3f}, w_fusion={learned_w2:.3f}, w_fnirs={learned_w3:.3f}")

    overfit_gap_train_vs_test = (train_acc * 100) - test_acc
    overfit_gap_val_vs_test = (best_val_acc * 100) - test_acc

    print(f"\n[Reference, not primary] Stage-2 final train_acc - test_acc: {overfit_gap_train_vs_test:+.2f}pp "
          f"(NOTE: Stage 2 has no validation signal, so this is expected to be large "
          f"by design and does NOT indicate real overfitting)")
    print(f"Generalization gap (best Stage-1 val_acc - test_acc): {overfit_gap_val_vs_test:+.2f} percentage points "
          f"(this is the meaningful overfitting/generalization indicator, "
          f"since val is now trial-level split with no leakage)")

    results = {
        'test_session': test_session,
        'train_sessions': train_sessions,
        'test_accuracy_final': test_acc,
        'kappa_final': kappa_final,
        'confusion_matrix_final': cm.tolist(),
        'ablation': {
            'eeg_only': eeg_acc,
            'fnirs_only': fnirs_acc,
            'fusion_only': fusion_acc,
            'final_weighted': test_acc,
        },
        'learned_weights': {
            'w_eeg': learned_w1,
            'w_fusion': learned_w2,
            'w_fnirs': learned_w3,
        },
        'best_val_acc': best_val_acc * 100,
        'final_train_acc': train_acc * 100,
        'overfitting_gap_train_vs_test': overfit_gap_train_vs_test,
        'overfitting_gap_val_vs_test': overfit_gap_val_vs_test,
    }
    results_path = os.path.join(PROCESSED_DATA_PATH, f'results_session{test_session}.json')
    with open(results_path, 'w') as f:
        json.dump(results, f, indent=2)
    print(f"\nResults saved to: {results_path}")

    train_dataset.cleanup()
    test_dataset.cleanup()

    del model
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    return results


if __name__ == "__main__":
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    print(f"Running on Device: {device}")
    print(f"Cross-Session Hold-Out: 3 runs (each session as test once)")
    print(f"All {len(ALL_SUBJECTS)} subjects used in each run")

    # QUICK TEST MODE: set QUICK_TEST=True to run only test_session=3 with a shorter Stage 2,
    # per the plan discussed with the user (patience stays 50 in Stage 1, only Stage 2 max
    # epochs is capped at 60 for a fast sanity check of the LSTM change vs baseline).
    QUICK_TEST = True          # NEW: flip to False for the full 3-session cross-session run
    QUICK_TEST_SESSIONS = [3]  # NEW: only test_session=3
    if QUICK_TEST:
        STAGE2_MAX_EPOCHS = 60  # NEW: overrides the module-level default for this run only
        sessions_to_run = QUICK_TEST_SESSIONS
        print(f"[QUICK TEST] STAGE2_MAX_EPOCHS overridden to {STAGE2_MAX_EPOCHS}, "
              f"running only test_session={QUICK_TEST_SESSIONS}")
    else:
        sessions_to_run = [1, 2, 3]

    all_session_results = []
    for test_session in sessions_to_run:
        result = run_training(weight_decay=0.0, device=device, test_session=test_session)
        all_session_results.append(result)

    print("\n" + "=" * 70)
    print("CROSS-SESSION HOLD-OUT SUMMARY")
    print("=" * 70)
    print(f"{'test_session':>14} | {'test_acc':>9} | {'best_val_acc':>13} | {'kappa':>8}")
    for r in all_session_results:
        print(f"{r['test_session']:>14} | {r['test_accuracy_final']:>8.2f}% | "
              f"{r['best_val_acc']:>12.2f}% | {r['kappa_final']:>8.4f}")

    final_acc   = np.mean([r['test_accuracy_final'] for r in all_session_results])
    final_kappa = np.mean([r['kappa_final']          for r in all_session_results])
    print(f"\nFinal average accuracy ({len(all_session_results)} session(s)): {final_acc:.2f}%")
    print(f"Final average Kappa   ({len(all_session_results)} session(s)): {final_kappa:.4f}")

    summary = {
        'session_results': all_session_results,
        'final_average_accuracy': final_acc,
        'final_average_kappa':   final_kappa,
    }
    summary_path = os.path.join(PROCESSED_DATA_PATH, 'cross_session_summary.json')
    with open(summary_path, 'w') as f:
        json.dump(summary, f, indent=2)
    print(f"\nFull cross-session summary saved to: {summary_path}")
    print("\nDone.")

Running on Device: cuda
Cross-Session Hold-Out: 3 runs (each session as test once)
All 26 subjects used in each run
[QUICK TEST] STAGE2_MAX_EPOCHS overridden to 60, running only test_session=[3]

######################################################################
# CROSS-SESSION: train on sessions [1, 2]  |  test on session 3
######################################################################
Scanning Train_Set - 26 subjects (sessions [1, 2])...
VP001 verified: 681 windows
VP002 verified: 684 windows
VP003 verified: 684 windows
VP004 verified: 453 windows
VP005 verified: 677 windows
VP006 verified: 684 windows
VP007 verified: 684 windows
VP008 verified: 684 windows
VP009 verified: 682 windows
VP010 verified: 684 windows
VP011 verified: 684 windows
VP012 verified: 684 windows
VP013 verified: 681 windows
VP014 verified: 674 windows
VP015 verified: 676 windows
VP016 verified: 684 windows
VP017 verified: 684 windows
VP018 verified: 684 windows
VP019 verified: 684 windows
VP020 verifi

In [ ]:
#تست بدون detach (isolation تک‌متغیره)

In [1]:
#Tsfnet cross session
#version4
#preprocess corrected with session_id
#subject-level val + lr=0.0001 + weight_decay=0.1 + label_smoothing + two dense layers+ WEIGHT_DECAY_GRID = [0.01, 0.02, 0.05]
#Tsfnet model run h 11segments
# This is comprehensive TSFNet code, please set the paths basede on your system
import os
import gc
import math
# import numpy as np
# FIX (Problem 3, part 1): 'random' module was used below (random.shuffle) but never imported,
# which would raise NameError: name 'random' is not defined the moment the script runs.
import random  # NEW: was missing, required for random.shuffle below
import numpy as np  # NEW: moved down here only for readability, same import as before
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import classification_report, accuracy_score
# NEW (user request): confusion_matrix needed for the detailed error-pattern analysis, and
# accuracy_score is reused per-branch for the EEG-only / fNIRS-only ablation below.
from sklearn.metrics import confusion_matrix  # NEW
from sklearn.metrics import cohen_kappa_score  # NEW: adopted from reviewing a third-party implementation -- corrects raw accuracy for chance agreement, a standard BCI-literature metric
import warnings
import json  # NEW: for results.json export, adopted from reviewing a third-party implementation
warnings.filterwarnings('ignore')

# CONFIGURATION
PROCESSED_DATA_PATH = r"F:\preprocessing-history-thesis\preproc_P2_no_fnirs_filter_session_id in code at the same time"  # FIX: was E:\, change to match your actual drive

ALL_SUBJECTS = list(range(1, 27))

RANDOM_SEED = 42  # fixed seed for reproducible model init and window-level val split
random.seed(RANDOM_SEED)
torch.manual_seed(RANDOM_SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False
np.random.seed(RANDOM_SEED)

class MMapHybridDataset(Dataset): 

    def __init__(self, subs_list, dataset_name="Dataset",
                 session_filter=None,        # CROSS-SESSION: None=همه sessions، [1,2]=فقط session 1 و 2
                 min_windows_threshold=500):
        super().__init__()
        self.dataset_name = dataset_name
        self.session_filter = session_filter  # CROSS-SESSION: کدام session‌ها بارگذاری شوند
        self.valid_files = []
        self.total_windows = 0
        
        filter_str = f"sessions {session_filter}" if session_filter is not None else "all sessions"
        print(f"Scanning {dataset_name} - {len(subs_list)} subjects ({filter_str})...")
        
        if not os.path.exists(PROCESSED_DATA_PATH):
            raise FileNotFoundError(
                f"\n[PATH ERROR] PROCESSED_DATA_PATH does not exist:\n"
                f"  '{PROCESSED_DATA_PATH}'\n"
                f"  Check your drive letter (D: vs E:) and folder structure.\n"
                f"  Expected subject folders: VP001, VP002, ... inside that directory."
            )
        
        for sub_id in subs_list:
            sub_str = f"{sub_id:03d}"
            subject = f"VP{sub_str}"
            sub_folder = os.path.join(PROCESSED_DATA_PATH, subject)
            
            if not os.path.exists(sub_folder):
                continue
                
            eeg_file = next((f for f in [
                os.path.join(sub_folder, f"{subject}_eeg_windows.npy"), 
                os.path.join(sub_folder, f"{subject}_eeg.npy")
            ] if os.path.exists(f)), None)
            
            nirs_file = next((f for f in [
                os.path.join(sub_folder, f"{subject}_fnirs_windows.npy"), 
                os.path.join(sub_folder, f"{subject}_fnirs.npy")
            ] if os.path.exists(f)), None)
            
            labels_file = os.path.join(sub_folder, f"{subject}_labels.npy")
            
            if eeg_file and nirs_file and os.path.exists(labels_file):
                try:
                    lbls = np.load(labels_file).astype(np.int64)

                    session_mask = None
                    if session_filter is not None:
                        session_ids_file = os.path.join(sub_folder, f"{subject}_session_ids.npy")
                        if os.path.exists(session_ids_file):
                            sess_ids = np.load(session_ids_file)
                            session_mask = np.isin(sess_ids, session_filter)
                            lbls = lbls[session_mask]
                        else:
                            print(f"  WARNING: {subject}_session_ids.npy not found — using all windows")

                    n_windows = lbls.shape[0]
                    
                    if n_windows < min_windows_threshold:
                        print(f"{subject} Skipped - Insufficient windows ({n_windows} < {min_windows_threshold}). Check task markers!")
                        continue
                    
                    self.valid_files.append({
                        'subject': subject,
                        'eeg_path': eeg_file,
                        'nirs_path': nirs_file,
                        'labels': lbls,
                        'session_mask': session_mask,
                        'start_idx': self.total_windows,
                        'end_idx': self.total_windows + n_windows
                    })
                    self.total_windows += n_windows
                    print(f"{subject} verified: {n_windows} windows")
                    
                except Exception as e:
                    print(f"Error scanning {subject}: {str(e)}")
                    continue
                    
        if self.total_windows == 0:
            raise ValueError(f"No valid preprocessed data found for {dataset_name} after filtering!")

        os.makedirs(PROCESSED_DATA_PATH, exist_ok=True)

        import re
        safe_slug = re.sub(r'[^a-z0-9]', '_', dataset_name.lower())[:12]
        self.mmap_eeg_path  = os.path.join(PROCESSED_DATA_PATH, f"t_mmap_{safe_slug}_e.npy")
        self.mmap_nirs_path = os.path.join(PROCESSED_DATA_PATH, f"t_mmap_{safe_slug}_n.npy")
        
        for f_path in [self.mmap_eeg_path, self.mmap_nirs_path]:
            if os.path.exists(f_path):
                try: os.remove(f_path)
                except: pass
            with open(f_path, 'wb') as f:
                pass 
        
        self.X_eeg = np.memmap(self.mmap_eeg_path, dtype='float32', mode='w+', shape=(self.total_windows, 16, 16, 600))
        self.X_nirs = np.memmap(self.mmap_nirs_path, dtype='float32', mode='w+', shape=(self.total_windows, 11, 16, 16, 30, 2))
        self.y = np.zeros(self.total_windows, dtype='int64')
        
        print(f"Streaming clean data to disk memory-map ({dataset_name})...")
        for f_info in self.valid_files:
            s, e = f_info['start_idx'], f_info['end_idx']
            eeg_full  = np.load(f_info['eeg_path'])
            nirs_full = np.load(f_info['nirs_path'])
            if f_info['session_mask'] is not None:
                eeg_full  = eeg_full[f_info['session_mask']]
                nirs_full = nirs_full[f_info['session_mask']]
            self.X_eeg[s:e]  = eeg_full
            self.X_nirs[s:e] = nirs_full
            self.y[s:e]      = f_info['labels']
            gc.collect()
            
        self.X_eeg.flush()
        self.X_nirs.flush()
        print(f"--> {dataset_name} Ready on Disk (CLEAN)! Total Windows: {self.total_windows} | Labels: {np.bincount(self.y)}\n")

    def __len__(self):
        return self.total_windows

    def __getitem__(self, idx):
        eeg_tensor = torch.from_numpy(self.X_eeg[idx]).float()
        nirs_tensor = torch.from_numpy(self.X_nirs[idx]).float()
        label_tensor = torch.tensor(self.y[idx], dtype=torch.long)
        return eeg_tensor, nirs_tensor, label_tensor
        
    def cleanup(self):
        del self.X_eeg
        del self.X_nirs
        gc.collect()
        for f in [self.mmap_eeg_path, self.mmap_nirs_path]:
            if os.path.exists(f):
                try: os.remove(f)
                except: pass

def pearson_correlation_loss(a, b, eps=1e-8):
    a_flat = a.reshape(-1)
    b_flat = b.reshape(-1)
    a_centered = a_flat - a_flat.mean()
    b_centered = b_flat - b_flat.mean()
    numerator = (a_centered * b_centered).sum()
    denominator = torch.sqrt((a_centered ** 2).sum() * (b_centered ** 2).sum()) + eps
    pcc = numerator / denominator
    return 1.0 - torch.abs(pcc)


class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=100):
        super().__init__()
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        self.register_buffer('pe', pe.unsqueeze(0))
    def forward(self, x): 
        return x + self.pe[:, :x.size(1), :]

class CrossAttention(nn.Module):
    def __init__(self, d_model=256, nhead=8, dropout=0.5, dim_feedforward=512):
        super().__init__()
        self.attn     = nn.MultiheadAttention(d_model, nhead, dropout=dropout, batch_first=True)
        self.norm1    = nn.LayerNorm(d_model)
        self.norm2    = nn.LayerNorm(d_model)
        self.dropout1 = nn.Dropout(dropout)
        self.dropout2 = nn.Dropout(dropout)
        self.ffn = nn.Sequential(
            nn.Linear(d_model, dim_feedforward),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(dim_feedforward, d_model),
        )
    def forward(self, query, key, value):
        attn_out, _ = self.attn(query, key, value)
        x = self.norm1(query + self.dropout1(attn_out))
        x = self.norm2(x + self.dropout2(self.ffn(x)))
        return x

class TSFNet_With_fNIRS_2D(nn.Module):
    def __init__(self, eeg_ch=28, num_classes=3, dropout=0.5):
        super(TSFNet_With_fNIRS_2D, self).__init__()

        # EEG branch
        self.eeg_conv1 = nn.Conv3d(1, 16, kernel_size=(12, 4, 4), stride=(6, 2, 2), padding=(3, 1, 1))
        self.eeg_bn1 = nn.BatchNorm3d(16)
        self.eeg_drop1 = nn.Dropout(dropout)
        self.eeg_conv2 = nn.Conv3d(16, 32, kernel_size=(6, 2, 2), stride=(2, 2, 2), padding=(2, 0, 0))
        self.eeg_bn2 = nn.BatchNorm3d(32)
        self.eeg_drop2 = nn.Dropout(dropout)
        self.eeg_fc = nn.Linear(32 * 4 * 4, 64)

        # NEW (Mughal et al. 2022-inspired Time-Distributed CNN-LSTM): replaces the plain temporal
        # GAP used for the EEG classifier head only. eeg_feat_map (mean over time) is still computed
        # separately and unchanged for L_EFGF -- this LSTM path is isolated and does not affect it.
        # input_size = flattened spatial dims of ex2 per timestep (32 channels * 4 * 4 spatial).
        self.eeg_temporal_lstm = nn.LSTM(
            input_size=32 * 4 * 4,
            hidden_size=64,
            num_layers=1,
            batch_first=True,
            bidirectional=False
        )  # NEW
        self.eeg_fc_from_lstm = nn.Linear(64, 64)  # NEW

        # fNIRS branch
        self.nirs_conv1 = nn.Conv3d(2, 16, kernel_size=(6, 4, 4), stride=(2, 2, 2), padding=(2, 1, 1))
        self.nirs_bn1 = nn.BatchNorm3d(16)
        self.nirs_drop1 = nn.Dropout(dropout)
        self.nirs_conv2 = nn.Conv3d(16, 32, kernel_size=(3, 2, 2), stride=(2, 2, 2), padding=(1, 0, 0))
        self.nirs_bn2 = nn.BatchNorm3d(32)
        self.nirs_drop2 = nn.Dropout(dropout)
        self.nirs_fc = nn.Linear(32 * 4 * 4, 64)

        # Fusion branch
        self.fus_conv1 = nn.Conv3d(1, 16, kernel_size=(12, 4, 4), stride=(6, 2, 2), padding=(3, 1, 1))
        self.fus_bn1   = nn.BatchNorm3d(16)
        self.fus_drop1 = nn.Dropout(dropout)
        self.fus_conv2 = nn.Conv3d(16, 32, kernel_size=(6, 2, 2), stride=(2, 2, 2), padding=(2, 0, 0))
        self.fus_bn2   = nn.BatchNorm3d(32)
        self.fus_drop2 = nn.Dropout(dropout)

        # EFGF two-stage
        self.efgf1_eeg_conv3d  = nn.Conv3d(16, 1, kernel_size=(6, 2, 2), stride=(1, 1, 1), padding=(0, 0, 0))
        self.efgf1_nirs_conv3d = nn.Conv3d(16, 1, kernel_size=(6, 2, 2), stride=(1, 1, 1), padding=(0, 0, 0))
        self.alpha1 = nn.Parameter(torch.tensor(0.0))
        self.gamma1 = nn.Parameter(torch.tensor(0.0))

        self.efgf2_eeg_conv3d  = nn.Conv3d(32, 1, kernel_size=(3, 2, 2), stride=(1, 1, 1), padding=(0, 0, 0))
        self.efgf2_nirs_conv3d = nn.Conv3d(32, 1, kernel_size=(3, 2, 2), stride=(1, 1, 1), padding=(0, 0, 0))
        self.alpha2 = nn.Parameter(torch.tensor(0.0))
        self.gamma2 = nn.Parameter(torch.tensor(0.0))

        self.fusion_proj = nn.Linear(32 * 4 * 4, 64)

        # CAFE
        CAFE_DIM = 128
        self.pos_encoder      = PositionalEncoding(CAFE_DIM, max_len=11)
        self.nirs_seq_proj    = nn.Linear(32, CAFE_DIM)
        self.fusion_cafe_proj = nn.Linear(64, CAFE_DIM)
        self.nirs_cafe_proj   = nn.Linear(64, CAFE_DIM)
        self.cafe_attn  = CrossAttention(CAFE_DIM, nhead=8, dropout=dropout, dim_feedforward=256)
        self.cafe_gate  = nn.Linear(CAFE_DIM * 2, CAFE_DIM)
        self.cafe_attn2 = CrossAttention(CAFE_DIM, nhead=8, dropout=dropout, dim_feedforward=256)
        self.cafe_gate2 = nn.Linear(CAFE_DIM * 2, CAFE_DIM)
        self.cafe_out_proj  = nn.Linear(CAFE_DIM, 64)
        self.cafe_out_proj2 = nn.Linear(CAFE_DIM, 64)

        # classifiers
        self.fc_eeg = nn.Sequential(
            nn.Linear(64, 32), nn.ELU(), nn.Dropout(dropout),
            nn.Linear(32, num_classes)
        )
        self.fc_fnirs = nn.Sequential(
            nn.Linear(64, 32), nn.ELU(), nn.Dropout(dropout),
            nn.Linear(32, num_classes)
        )
        self.fc_fusion = nn.Sequential(
            nn.Linear(64, 32), nn.ELU(), nn.Dropout(dropout),
            nn.Linear(32, num_classes)
        )

        self.w1 = nn.Parameter(torch.tensor(1.0))
        self.w2 = nn.Parameter(torch.tensor(1.0))
        self.w3 = nn.Parameter(torch.tensor(1.0))
        self.dropout = nn.Dropout(dropout)
        
    def forward(self, eeg, nirs):
        batch_size = eeg.size(0)

        # EEG branch
        eeg_in = eeg.permute(0, 3, 1, 2).unsqueeze(1)       # (batch,1,600,16,16)
        ex1 = F.elu(self.eeg_bn1(self.eeg_conv1(eeg_in)))   # (batch,16,100,8,8)
        ex1 = self.eeg_drop1(ex1)
        ex2 = F.elu(self.eeg_bn2(self.eeg_conv2(ex1)))      # (batch,32,50,4,4)
        ex2 = self.eeg_drop2(ex2)

        # GAP path (unchanged, isolated) -- used ONLY for L_EFGF, not for the EEG classifier anymore
        eeg_feat_map = ex2.mean(dim=2)                       # (batch,32,4,4) — برای L_EFGF فقط

        # NEW: LSTM path for EEG classifier head — replaces GAP-based eeg_feat.
        # .detach() on ex2 ensures eeg_conv1/eeg_conv2 receive NO gradient from this LSTM path;
        # they are only trained through the fusion branch / EFGF path, exactly like eeg_feat_map
        # above and consistent with the prior SpatialTransformer isolation lesson.
        batch_size_eeg = ex2.size(0)
        T_dim = ex2.size(2)
        #eeg_seq = ex2.detach().permute(0, 2, 1, 3, 4).reshape(batch_size_eeg, T_dim, -1)  # NEW: (batch,T,512)
        eeg_seq = ex2.permute(0, 2, 1, 3, 4).reshape(batch_size_eeg, T_dim, -1)  # detach حذف شد
        lstm_out, (h_n, c_n) = self.eeg_temporal_lstm(eeg_seq)  # NEW
        eeg_feat_from_lstm = h_n.squeeze(0)                      # NEW: (batch,64)
        eeg_feat = self.eeg_fc_from_lstm(eeg_feat_from_lstm)     # NEW: (batch,64)

        # Fusion branch — Conv1 (قبل از EFGF1)
        fx1 = F.elu(self.fus_bn1(self.fus_conv1(eeg_in)))   # (batch,16,100,8,8)
        fx1 = self.fus_drop1(fx1)

        # fNIRS branch
        n_seg  = nirs.size(1)
        nirs_r = nirs.permute(0, 1, 5, 4, 2, 3)
        nirs_r = nirs_r.reshape(batch_size * n_seg, 2, 30, 16, 16)
        ny1 = F.elu(self.nirs_bn1(self.nirs_conv1(nirs_r)))
        ny1 = self.nirs_drop1(ny1)
        ny2 = F.elu(self.nirs_bn2(self.nirs_conv2(ny1)))
        ny2 = self.nirs_drop2(ny2)
        ny1 = ny1.reshape(batch_size, n_seg, 16, 15, 8, 8)
        ny2 = ny2.reshape(batch_size, n_seg, 32,  8, 4, 4)
        nirs_feat_map = ny2.mean(dim=(1, 3))
        nirs_feat = self.nirs_fc(nirs_feat_map.flatten(1))
        y_spatial = ny2.mean(dim=(3, 4, 5))

        # EFGF1
        ny1_vol = ny1.mean(dim=1)
        eeg_a1_vol  = self.efgf1_eeg_conv3d(ex1)
        nirs_a1_vol = self.efgf1_nirs_conv3d(ny1_vol)
        eeg_a1  = torch.sigmoid(eeg_a1_vol.mean(dim=2))
        nirs_a1 = torch.sigmoid(nirs_a1_vol.mean(dim=2))
        a1   = torch.clamp(self.alpha1, 0, 1)
        Phi1 = a1 * nirs_a1 + (1 - a1) * eeg_a1
        g1 = torch.clamp(self.gamma1, 0, 1)
        Phi1_bc = Phi1.unsqueeze(2)
        fx1_updated = g1 * ex1[:, :, :, :Phi1.shape[-2], :Phi1.shape[-1]] \
                    + (1 - g1) * fx1[:, :, :, :Phi1.shape[-2], :Phi1.shape[-1]] \
                    + Phi1_bc * fx1[:, :, :, :Phi1.shape[-2], :Phi1.shape[-1]]

        if fx1_updated.shape != fx1.shape:
            fx1_updated_aligned = F.adaptive_avg_pool3d(
                fx1_updated, (fx1.shape[2], fx1.shape[3], fx1.shape[4])
            )
        else:
            fx1_updated_aligned = fx1_updated
        fx2 = F.elu(self.fus_bn2(self.fus_conv2(fx1_updated_aligned)))
        fx2 = self.fus_drop2(fx2)

        # EFGF2
        ny2_vol = ny2.mean(dim=1)
        eeg_a2_vol  = self.efgf2_eeg_conv3d(ex2)
        nirs_a2_vol = self.efgf2_nirs_conv3d(ny2_vol)
        eeg_a2  = torch.sigmoid(eeg_a2_vol.mean(dim=2))
        nirs_a2 = torch.sigmoid(nirs_a2_vol.mean(dim=2))
        a2   = torch.clamp(self.alpha2, 0, 1)
        Phi2 = a2 * nirs_a2 + (1 - a2) * eeg_a2
        g2   = torch.clamp(self.gamma2, 0, 1)
        Phi2_bc = Phi2.unsqueeze(2)
        sh, sw = Phi2.shape[-2], Phi2.shape[-1]
        fx2_updated = g2 * ex2[:, :, :, :sh, :sw] \
                    + (1 - g2) * fx2[:, :, :, :sh, :sw] \
                    + Phi2_bc * fx2[:, :, :, :sh, :sw]
        fus_feat_map = fx2_updated.mean(dim=2)
        fus_feat_map = F.adaptive_avg_pool2d(fus_feat_map, (4, 4))
        fused_feat   = self.fusion_proj(fus_feat_map.flatten(1))

        Phi2_bc_loss = Phi2.expand(-1, 32, -1, -1)
        if Phi2_bc_loss.shape != eeg_feat_map.shape:
            Phi2_bc_loss = F.adaptive_avg_pool2d(Phi2_bc_loss, eeg_feat_map.shape[-2:])

        # CAFE
        nirs_seq_256  = self.nirs_seq_proj(y_spatial)
        nirs_proj_256 = self.pos_encoder(nirs_seq_256)
        fused_256     = self.fusion_cafe_proj(fused_feat)
        nirs_feat_256 = self.nirs_cafe_proj(nirs_feat)
        fused_seq_256  = fused_256.unsqueeze(1)
        enhanced_256   = self.cafe_attn(fused_seq_256, nirs_proj_256, nirs_proj_256).squeeze(1)
        gate2_256      = torch.sigmoid(self.cafe_gate(torch.cat([fused_256, enhanced_256], dim=-1)))
        final_feat_256 = gate2_256 * enhanced_256 + (1 - gate2_256) * fused_256
        nirs_q_256     = nirs_feat_256.unsqueeze(1)
        fused_kv_256   = fused_256.unsqueeze(1)
        enh_nirs_256   = self.cafe_attn2(nirs_q_256, fused_kv_256, fused_kv_256).squeeze(1)
        gate3_256      = torch.sigmoid(self.cafe_gate2(torch.cat([nirs_feat_256, enh_nirs_256], dim=-1)))
        final_nirs_256 = gate3_256 * enh_nirs_256 + (1 - gate3_256) * nirs_feat_256
        final_feat = self.cafe_out_proj(final_feat_256)
        final_nirs = self.cafe_out_proj2(final_nirs_256)

        # Classifier
        eeg_out    = self.fc_eeg(self.dropout(eeg_feat))
        fnirs_out  = self.fc_fnirs(self.dropout(final_nirs))
        fusion_out = self.fc_fusion(self.dropout(final_feat))

        w1 = torch.sigmoid(self.w1)
        w2 = torch.sigmoid(self.w2)
        w3 = torch.sigmoid(self.w3)
        final_out = w1 * eeg_out + w2 * fusion_out + w3 * fnirs_out

        return (final_out, eeg_out, fnirs_out, fusion_out,
                eeg_feat_map, Phi2_bc_loss,
                final_nirs,
                final_feat)


LAMBDA_EEG = 0.2
LAMBDA_FNIRS = 0.2
LAMBDA_EFGF_REG = 1.0
LAMBDA_CAFE_REG = 1.0

STAGE1_MAX_EPOCHS = 300
STAGE1_PATIENCE   = 50
STAGE2_MAX_EPOCHS = 200


def run_training(weight_decay, device, test_session=3):
    train_sessions = [s for s in [1, 2, 3] if s != test_session]
    print("\n" + "#" * 70)
    print(f"# CROSS-SESSION: train on sessions {train_sessions}  |  test on session {test_session}")
    print("#" * 70)

    from torch.utils.data import SubsetRandomSampler

    train_dataset = MMapHybridDataset(ALL_SUBJECTS, "Train_Set",
                                       session_filter=train_sessions,
                                       min_windows_threshold=200)
    test_dataset  = MMapHybridDataset(ALL_SUBJECTS, "Test_Set",
                                       session_filter=[test_session],
                                       min_windows_threshold=100)

    all_trial_ids_global = []
    window_offset = 0
    for f_info in train_dataset.valid_files:
        subject = f_info['subject']
        sub_folder = os.path.join(PROCESSED_DATA_PATH, subject)
        trial_ids_path = os.path.join(sub_folder, f"{subject}_trial_ids.npy")
        
        trial_ids_full = np.load(trial_ids_path)
        
        if f_info['session_mask'] is not None:
            trial_ids_filtered = trial_ids_full[f_info['session_mask']]
        else:
            trial_ids_filtered = trial_ids_full
        
        n_windows_this_subject = f_info['end_idx'] - f_info['start_idx']
        assert len(trial_ids_filtered) == n_windows_this_subject, \
            f"{subject}: trial_ids mismatch ({len(trial_ids_filtered)} vs {n_windows_this_subject})"
        
        sub_idx = ALL_SUBJECTS.index(int(subject[2:]))
        global_trial_ids = sub_idx * 100 + trial_ids_filtered
        all_trial_ids_global.extend(global_trial_ids.tolist())
        window_offset += n_windows_this_subject
    
    all_trial_ids_global = np.array(all_trial_ids_global)
    assert len(all_trial_ids_global) == len(train_dataset), \
        f"trial_ids length mismatch: {len(all_trial_ids_global)} vs {len(train_dataset)}"
    
    unique_trials = np.unique(all_trial_ids_global)
    rng = np.random.default_rng(RANDOM_SEED)
    rng.shuffle(unique_trials)
    
    split_trial = int(len(unique_trials) * 0.8)
    train_trials = set(unique_trials[:split_trial])
    val_trials   = set(unique_trials[split_trial:])
    
    s1_train_indices = [i for i, t in enumerate(all_trial_ids_global) if t in train_trials]
    s1_val_indices   = [i for i, t in enumerate(all_trial_ids_global) if t in val_trials]
    
    print(f"[Trial-level split] {len(unique_trials)} trials total | "
          f"{len(train_trials)} train trials ({len(s1_train_indices)} windows) | "
          f"{len(val_trials)} val trials ({len(s1_val_indices)} windows) | "
          f"ratio={len(s1_train_indices)/max(len(s1_val_indices),1):.1f}:1")
    print(f"[Trial-level split] Zero overlap guaranteed: "
          f"no window from a val trial appears in train")

    s1_train_loader = DataLoader(
        train_dataset, batch_size=64,
        sampler=SubsetRandomSampler(s1_train_indices),
        num_workers=0
    )
    s1_val_loader = DataLoader(
        train_dataset, batch_size=64,
        sampler=SubsetRandomSampler(s1_val_indices),
        num_workers=0
    )
    train_loader  = DataLoader(train_dataset, batch_size=64, shuffle=True,  num_workers=0)
    test_loader   = DataLoader(test_dataset,  batch_size=64, shuffle=False, num_workers=0)

    model = TSFNet_With_fNIRS_2D(eeg_ch=28, num_classes=3, dropout=0.5).to(device)

    print(f"Model Initialized. Total Parameters: {sum(p.numel() for p in model.parameters()):,}")
    print(f"Running on Device: {device}\n")

    optimizer = optim.Adam(model.parameters(), lr=0.001)

    _train_labels   = train_dataset.y
    _class_counts   = np.bincount(_train_labels, minlength=3).astype(float)
    _class_weights  = 1.0 / (_class_counts + 1e-6)
    _class_weights  = _class_weights / _class_weights.sum() * 3
    _weight_tensor  = torch.tensor(_class_weights, dtype=torch.float32).to(device)
    print(f"[3-class] Class counts (train): 0-back={int(_class_counts[0])}, "
          f"2-back={int(_class_counts[1])}, 3-back={int(_class_counts[2])}")
    print(f"[3-class] Class weights:        0-back={_class_weights[0]:.3f}, "
          f"2-back={_class_weights[1]:.3f}, 3-back={_class_weights[2]:.3f}")
    criterion = nn.CrossEntropyLoss(weight=_weight_tensor)

    ckpt_path = os.path.join(PROCESSED_DATA_PATH, f'best_tsfnet_session{test_session}.pth')

    print(f"STAGE 1 — training on {len(s1_train_indices)} windows (80% of train sessions {train_sessions}), "
          f"val on {len(s1_val_indices)} windows (20%) | "
          f"max {STAGE1_MAX_EPOCHS} epochs | patience {STAGE1_PATIENCE} | trial-level 4:1 split (paper)")

    best_val_acc     = 0.0
    epochs_no_improve = 0
    stage1_best_val_train_loss = float('inf')
    stage1_final_train_loss    = float('inf')
    train_acc = 0.0

    for epoch in range(1, STAGE1_MAX_EPOCHS + 1):
        model.train()
        train_loss, train_class_loss, train_efgf_loss, train_cafe_loss, train_correct = 0.0, 0.0, 0.0, 0.0, 0

        for eeg_b, nirs_b, lbl_b in s1_train_loader:
            eeg_b, nirs_b, lbl_b = eeg_b.to(device), nirs_b.to(device), lbl_b.to(device)
            optimizer.zero_grad()
            pred, eeg_out, fnirs_out, fusion_out, eeg_feat_map, efgf_attn_map, cafe_fnirs_feat, cafe_fusion_feat = model(eeg_b, nirs_b)

            L_pred   = criterion(pred,      lbl_b)
            L_eeg    = criterion(eeg_out,   lbl_b)
            L_fnirs  = criterion(fnirs_out, lbl_b)
            L_fusion = criterion(fusion_out,lbl_b)
            L_class  = L_pred + LAMBDA_EEG * L_eeg + LAMBDA_FNIRS * L_fnirs + L_fusion
            L_efgf   = pearson_correlation_loss(eeg_feat_map, efgf_attn_map)
            L_cafe   = pearson_correlation_loss(cafe_fnirs_feat, cafe_fusion_feat)
            loss     = L_class + LAMBDA_EFGF_REG * L_efgf + LAMBDA_CAFE_REG * L_cafe

            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()

            train_loss       += loss.item()
            train_class_loss += L_class.item()
            train_efgf_loss  += L_efgf.item()
            train_cafe_loss  += L_cafe.item()
            train_correct    += (pred.argmax(1) == lbl_b).sum().item()

        train_acc      = train_correct / len(s1_train_indices)
        avg_train_loss = train_loss / len(s1_train_loader)
        stage1_final_train_loss = avg_train_loss

        model.eval()
        val_correct = 0
        with torch.no_grad():
            for eeg_b, nirs_b, lbl_b in s1_val_loader:
                eeg_b, nirs_b, lbl_b = eeg_b.to(device), nirs_b.to(device), lbl_b.to(device)
                pred, _, _, _, _, _, _, _ = model(eeg_b, nirs_b)
                val_correct += (pred.argmax(1) == lbl_b).sum().item()
        val_acc = val_correct / len(s1_val_indices)

        if val_acc > best_val_acc:
            best_val_acc = val_acc
            stage1_best_val_train_loss = avg_train_loss
            torch.save(model.state_dict(), ckpt_path)
            epochs_no_improve = 0
        else:
            epochs_no_improve += 1

        if epoch % 5 == 0 or epoch == 1:
            n_batches = len(s1_train_loader)
            print(f"[S1] Epoch {epoch:3d}/{STAGE1_MAX_EPOCHS} | Loss: {avg_train_loss:.4f} | "
                  f"Class: {train_class_loss/n_batches:.4f} | "
                  f"EFGF: {train_efgf_loss/n_batches:.4f} | "
                  f"CAFE: {train_cafe_loss/n_batches:.4f} | "
                  f"Train Acc: {train_acc*100:.2f}% | Val Acc: {val_acc*100:.2f}% | Best Val: {best_val_acc*100:.2f}%")

        if epochs_no_improve >= STAGE1_PATIENCE:
            print(f"\n[S1] Early stopping at epoch {epoch} "
                  f"(no val improvement for {STAGE1_PATIENCE} epochs). "
                  f"Best Val: {best_val_acc*100:.2f}%")
            break

    model.load_state_dict(torch.load(ckpt_path))
    stage2_target = min(stage1_best_val_train_loss, stage1_final_train_loss)
    print(f"\n[S1] Best checkpoint restored.")
    print(f"[S1] best-val-epoch loss: {stage1_best_val_train_loss:.4f} | final-epoch loss: {stage1_final_train_loss:.4f}")
    print(f"[S1] Stage-2 target loss: {stage2_target:.4f} (min of the two — ensures stage 2 actually trains)")

    print(f"\nSTAGE 2 — retraining on full {len(train_dataset)} samples ({len(ALL_SUBJECTS)} subjects, sessions {train_sessions}) "
          f"| max {STAGE2_MAX_EPOCHS} epochs | target loss < {stage2_target:.4f}")

    optimizer2 = optim.Adam(model.parameters(), lr=0.001)

    for epoch in range(1, STAGE2_MAX_EPOCHS + 1):
        model.train()
        s2_loss, s2_correct = 0.0, 0
        s2_class_loss = s2_efgf_loss = s2_cafe_loss = 0.0

        for eeg_b, nirs_b, lbl_b in train_loader:
            eeg_b, nirs_b, lbl_b = eeg_b.to(device), nirs_b.to(device), lbl_b.to(device)
            optimizer2.zero_grad()
            pred, eeg_out, fnirs_out, fusion_out, eeg_feat_map, efgf_attn_map, cafe_fnirs_feat, cafe_fusion_feat = model(eeg_b, nirs_b)

            L_pred   = criterion(pred,      lbl_b)
            L_eeg    = criterion(eeg_out,   lbl_b)
            L_fnirs  = criterion(fnirs_out, lbl_b)
            L_fusion = criterion(fusion_out,lbl_b)
            L_class  = L_pred + LAMBDA_EEG * L_eeg + LAMBDA_FNIRS * L_fnirs + L_fusion
            L_efgf   = pearson_correlation_loss(eeg_feat_map, efgf_attn_map)
            L_cafe   = pearson_correlation_loss(cafe_fnirs_feat, cafe_fusion_feat)
            loss     = L_class + LAMBDA_EFGF_REG * L_efgf + LAMBDA_CAFE_REG * L_cafe

            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer2.step()

            s2_loss       += loss.item()
            s2_class_loss += L_class.item()
            s2_efgf_loss  += L_efgf.item()
            s2_cafe_loss  += L_cafe.item()
            s2_correct    += (pred.argmax(1) == lbl_b).sum().item()

        avg_s2_loss = s2_loss / len(train_loader)
        train_acc   = s2_correct / len(train_loader.dataset)

        if epoch % 5 == 0 or epoch == 1:
            n_b = len(train_loader)
            print(f"[S2] Epoch {epoch:3d}/{STAGE2_MAX_EPOCHS} | Loss: {avg_s2_loss:.4f} | "
                  f"Class: {s2_class_loss/n_b:.4f} | "
                  f"EFGF: {s2_efgf_loss/n_b:.4f} | "
                  f"CAFE: {s2_cafe_loss/n_b:.4f} | "
                  f"Train Acc: {train_acc*100:.2f}% | Target loss: {stage2_target:.4f}")

        if avg_s2_loss < stage2_target:
            print(f"\n[S2] Stopping at epoch {epoch}: train loss {avg_s2_loss:.4f} "
                  f"< target {stage2_target:.4f}")
            torch.save(model.state_dict(), ckpt_path)
            break
    else:
        torch.save(model.state_dict(), ckpt_path)
        print(f"[S2] Reached max epochs ({STAGE2_MAX_EPOCHS}) without meeting loss target. Saving final weights.")

    print(f"\n RUNNING FINAL TEST EVALUATION (test_session={test_session})")
    model.load_state_dict(torch.load(ckpt_path))
    model.eval()

    all_preds_final, all_preds_eeg, all_preds_fnirs, all_preds_fusion, all_true = [], [], [], [], []
    with torch.no_grad():
        for eeg_b, nirs_b, lbl_b in test_loader:
            eeg_b, nirs_b, lbl_b = eeg_b.to(device), nirs_b.to(device), lbl_b.to(device)
            pred, eeg_out, fnirs_out, fusion_out, _, _, _, _ = model(eeg_b, nirs_b)
            all_preds_final.extend(pred.argmax(1).cpu().numpy())
            all_preds_eeg.extend(eeg_out.argmax(1).cpu().numpy())
            all_preds_fnirs.extend(fnirs_out.argmax(1).cpu().numpy())
            all_preds_fusion.extend(fusion_out.argmax(1).cpu().numpy())
            all_true.extend(lbl_b.cpu().numpy())

    class_names = ['0-back', '2-back', '3-back']

    test_acc = accuracy_score(all_true, all_preds_final) * 100
    kappa_final = cohen_kappa_score(all_true, all_preds_final)
    print(f"\nOverall Model Test Accuracy (final fused prediction): {test_acc:.2f}%")
    print(f"Kappa Score (final fused prediction): {kappa_final:.4f}")
    print("\nClassification Report (final fused prediction):")
    print(classification_report(all_true, all_preds_final, target_names=class_names))

    cm = confusion_matrix(all_true, all_preds_final)
    print("\nConfusion Matrix (rows=true, cols=predicted):")
    header = "        " + "".join([f"{c:>10}" for c in class_names])
    print(header)
    for i, row in enumerate(cm):
        row_str = "".join([f"{v:>10}" for v in row])
        print(f"{class_names[i]:>8}{row_str}")

    eeg_acc = accuracy_score(all_true, all_preds_eeg) * 100
    fnirs_acc = accuracy_score(all_true, all_preds_fnirs) * 100
    fusion_acc = accuracy_score(all_true, all_preds_fusion) * 100
    print("\nZero-Retraining Ablation (individual classifier heads of the SAME trained model):")
    print(f"  EEG-branch-only accuracy:          {eeg_acc:.2f}%")
    print(f"  fNIRS-branch-only accuracy:         {fnirs_acc:.2f}%")
    print(f"  EFGF+CAFE fused-feature accuracy:   {fusion_acc:.2f}%")
    print(f"  Final weighted-combination accuracy: {test_acc:.2f}%  (Mean(w1*eeg, w2*fusion, w3*fnirs))")

    with torch.no_grad():
        learned_w1 = torch.sigmoid(model.w1).item()
        learned_w2 = torch.sigmoid(model.w2).item()
        learned_w3 = torch.sigmoid(model.w3).item()
    print(f"\nLearned final-combination weights (after sigmoid, paper Eq.4): "
          f"w_eeg={learned_w1:.3f}, w_fusion={learned_w2:.3f}, w_fnirs={learned_w3:.3f}")

    overfit_gap_train_vs_test = (train_acc * 100) - test_acc
    overfit_gap_val_vs_test = (best_val_acc * 100) - test_acc

    print(f"\n[Reference, not primary] Stage-2 final train_acc - test_acc: {overfit_gap_train_vs_test:+.2f}pp "
          f"(NOTE: Stage 2 has no validation signal, so this is expected to be large "
          f"by design and does NOT indicate real overfitting)")
    print(f"Generalization gap (best Stage-1 val_acc - test_acc): {overfit_gap_val_vs_test:+.2f} percentage points "
          f"(this is the meaningful overfitting/generalization indicator, "
          f"since val is now trial-level split with no leakage)")

    results = {
        'test_session': test_session,
        'train_sessions': train_sessions,
        'test_accuracy_final': test_acc,
        'kappa_final': kappa_final,
        'confusion_matrix_final': cm.tolist(),
        'ablation': {
            'eeg_only': eeg_acc,
            'fnirs_only': fnirs_acc,
            'fusion_only': fusion_acc,
            'final_weighted': test_acc,
        },
        'learned_weights': {
            'w_eeg': learned_w1,
            'w_fusion': learned_w2,
            'w_fnirs': learned_w3,
        },
        'best_val_acc': best_val_acc * 100,
        'final_train_acc': train_acc * 100,
        'overfitting_gap_train_vs_test': overfit_gap_train_vs_test,
        'overfitting_gap_val_vs_test': overfit_gap_val_vs_test,
    }
    results_path = os.path.join(PROCESSED_DATA_PATH, f'results_session{test_session}.json')
    with open(results_path, 'w') as f:
        json.dump(results, f, indent=2)
    print(f"\nResults saved to: {results_path}")

    train_dataset.cleanup()
    test_dataset.cleanup()

    del model
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    return results


if __name__ == "__main__":
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    print(f"Running on Device: {device}")
    print(f"Cross-Session Hold-Out: 3 runs (each session as test once)")
    print(f"All {len(ALL_SUBJECTS)} subjects used in each run")

    # QUICK TEST MODE: set QUICK_TEST=True to run only test_session=3 with a shorter Stage 2,
    # per the plan discussed with the user (patience stays 50 in Stage 1, only Stage 2 max
    # epochs is capped at 60 for a fast sanity check of the LSTM change vs baseline).
    QUICK_TEST = True          # NEW: flip to False for the full 3-session cross-session run
    QUICK_TEST_SESSIONS = [3]  # NEW: only test_session=3
    if QUICK_TEST:
        STAGE2_MAX_EPOCHS = 60  # NEW: overrides the module-level default for this run only
        sessions_to_run = QUICK_TEST_SESSIONS
        print(f"[QUICK TEST] STAGE2_MAX_EPOCHS overridden to {STAGE2_MAX_EPOCHS}, "
              f"running only test_session={QUICK_TEST_SESSIONS}")
    else:
        sessions_to_run = [1, 2, 3]

    all_session_results = []
    for test_session in sessions_to_run:
        result = run_training(weight_decay=0.0, device=device, test_session=test_session)
        all_session_results.append(result)

    print("\n" + "=" * 70)
    print("CROSS-SESSION HOLD-OUT SUMMARY")
    print("=" * 70)
    print(f"{'test_session':>14} | {'test_acc':>9} | {'best_val_acc':>13} | {'kappa':>8}")
    for r in all_session_results:
        print(f"{r['test_session']:>14} | {r['test_accuracy_final']:>8.2f}% | "
              f"{r['best_val_acc']:>12.2f}% | {r['kappa_final']:>8.4f}")

    final_acc   = np.mean([r['test_accuracy_final'] for r in all_session_results])
    final_kappa = np.mean([r['kappa_final']          for r in all_session_results])
    print(f"\nFinal average accuracy ({len(all_session_results)} session(s)): {final_acc:.2f}%")
    print(f"Final average Kappa   ({len(all_session_results)} session(s)): {final_kappa:.4f}")

    summary = {
        'session_results': all_session_results,
        'final_average_accuracy': final_acc,
        'final_average_kappa':   final_kappa,
    }
    summary_path = os.path.join(PROCESSED_DATA_PATH, 'cross_session_summary.json')
    with open(summary_path, 'w') as f:
        json.dump(summary, f, indent=2)
    print(f"\nFull cross-session summary saved to: {summary_path}")
    print("\nDone.")

Running on Device: cuda
Cross-Session Hold-Out: 3 runs (each session as test once)
All 26 subjects used in each run
[QUICK TEST] STAGE2_MAX_EPOCHS overridden to 60, running only test_session=[3]

######################################################################
# CROSS-SESSION: train on sessions [1, 2]  |  test on session 3
######################################################################
Scanning Train_Set - 26 subjects (sessions [1, 2])...
VP001 verified: 681 windows
VP002 verified: 684 windows
VP003 verified: 684 windows
VP004 verified: 453 windows
VP005 verified: 677 windows
VP006 verified: 684 windows
VP007 verified: 684 windows
VP008 verified: 684 windows
VP009 verified: 682 windows
VP010 verified: 684 windows
VP011 verified: 684 windows
VP012 verified: 684 windows
VP013 verified: 681 windows
VP014 verified: 674 windows
VP015 verified: 676 windows
VP016 verified: 684 windows
VP017 verified: 684 windows
VP018 verified: 684 windows
VP019 verified: 684 windows
VP020 verifi